# Prepare ArtBench images locally (with updated CSV)

This notebook will do 3 things:
1. Read your original `ArtBench-10 (1).csv` file.
2. Download each image into local folders by style label (like your WikiArt structure).
3. Create **new CSV files** with local paths, without changing the original CSV.

Everything is written in simple steps, so you can run cell by cell.

## Step 1 — Imports and paths

This cell:
- imports the tools we need,
- finds the ArtBench CSV file,
- sets output locations for images and new CSV files.

In [2]:
from pathlib import Path
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests
import re
import time

def find_artbench_csv() -> Path:
    candidates = [
        Path('datasets/Artbench/ArtBench-10 (1).csv'),
        Path('ArtBench-10 (1).csv'),
        Path('../Artbench/ArtBench-10 (1).csv'),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    matches = list(Path.cwd().glob('**/ArtBench-10 (1).csv'))
    if matches:
        return matches[0].resolve()

    raise FileNotFoundError(
        'Could not find ArtBench-10 (1).csv. Open this notebook from the project root or datasets/Artbench.'
    )

CSV_PATH = find_artbench_csv()
DATASET_DIR = CSV_PATH.parent
IMAGES_DIR = DATASET_DIR / 'images'
OUTPUT_CSV = DATASET_DIR / 'ArtBench-10-local-paths.csv'
SIMPLE_OUTPUT_CSV = DATASET_DIR / 'ArtBench-10-local-simple.csv'
FAILURES_CSV = DATASET_DIR / 'ArtBench-10-download-failures.csv'

MAX_WORKERS = 16
REQUEST_TIMEOUT = 25
MAX_RETRIES = 3

print(f'CSV_PATH: {CSV_PATH}')
print(f'IMAGES_DIR: {IMAGES_DIR}')
print(f'OUTPUT_CSV: {OUTPUT_CSV}')

CSV_PATH: C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10 (1).csv
IMAGES_DIR: C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\images
OUTPUT_CSV: C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10-local-paths.csv


## Step 2 — Read the CSV

We load the data and quickly check important columns (`name`, `url`, `label`).

In [3]:
df = pd.read_csv(CSV_PATH)
required_columns = {'name', 'url', 'label'}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

print(f'Rows: {len(df):,}')
print(f'Unique labels: {df["label"].nunique()}')
display(df.head(3))
display(df['label'].value_counts())

Rows: 60,000
Unique labels: 10


,name,artist,url,is_public_domain,length,width,label,split,cifar_index
0,frank-omeara_towards-night-and-winter.jpg,frank-omeara,https://uploads5.wikiart.org/00316/images/fran...,True,800,657,impressionism,train,43186
1,goldstein-grigoriy_morning.jpg,goldstein-grigoriy,https://uploads5.wikiart.org/images/grigoriy-g...,True,521,499,impressionism,train,41151
2,georges-lemmen_man-reading.jpg,georges-lemmen,https://uploads6.wikiart.org/images/georges-le...,True,800,612,impressionism,train,9754


label
impressionism         6000
romanticism           6000
expressionism         6000
surrealism            6000
art_nouveau           6000
renaissance           6000
realism               6000
post_impressionism    6000
baroque               6000
ukiyo_e               6000
Name: count, dtype: int64

## Step 3 — Helper functions

These functions:
- create safe folder/file names,
- download one image with retries,
- return both success status and local path.

In [4]:
def sanitize_folder_name(value: str) -> str:
    text = str(value).strip()
    text = text.replace(' ', '_')
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text or 'unknown'


def pick_filename(row_name: str, url: str) -> str:
    row_name = str(row_name).strip()
    if row_name and Path(row_name).suffix:
        return Path(row_name).name

    url_path_name = Path(unquote(urlparse(str(url)).path)).name
    if url_path_name and Path(url_path_name).suffix:
        return url_path_name

    return 'image.jpg'


def download_one(index: int, row: dict):
    label_folder = sanitize_folder_name(row.get('label', 'unknown'))
    filename = pick_filename(row.get('name', ''), row.get('url', ''))

    target_dir = IMAGES_DIR / label_folder
    target_dir.mkdir(parents=True, exist_ok=True)
    target_file = target_dir / filename

    local_rel = target_file.relative_to(DATASET_DIR).as_posix()

    if target_file.exists() and target_file.stat().st_size > 0:
        return {
            'index': index,
            'ok': True,
            'local_path': local_rel,
            'error': ''
        }

    url = str(row.get('url', '')).strip()
    if not url:
        return {
            'index': index,
            'ok': False,
            'local_path': '',
            'error': 'Empty URL'
        }

    last_error = ''
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, timeout=REQUEST_TIMEOUT, stream=True)
            response.raise_for_status()

            with open(target_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 32):
                    if chunk:
                        f.write(chunk)

            if target_file.stat().st_size == 0:
                raise ValueError('Downloaded file is empty')

            return {
                'index': index,
                'ok': True,
                'local_path': local_rel,
                'error': ''
            }
        except Exception as exc:
            last_error = str(exc)
            if attempt < MAX_RETRIES:
                time.sleep(1.0)

    return {
        'index': index,
        'ok': False,
        'local_path': '',
        'error': last_error
    }

## Step 4 — Download all images into label folders

This can take a while because there are many images.

Good to know:
- It is resumable: already downloaded files are skipped.
- It prints progress while running.

In [5]:
records = df.to_dict(orient='records')
total = len(records)
results = []

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_one, i, row): i for i, row in enumerate(records)}

    for done_count, future in enumerate(as_completed(futures), start=1):
        results.append(future.result())

        if done_count % 250 == 0 or done_count == total:
            print(f'Processed {done_count:,}/{total:,}')

result_df = pd.DataFrame(results).sort_values('index')

success_count = int(result_df['ok'].sum())
fail_count = int((~result_df['ok']).sum())

print(f'✅ Success: {success_count:,}')
print(f'❌ Failed : {fail_count:,}')

Processed 250/60,000
Processed 500/60,000
Processed 750/60,000
Processed 1,000/60,000
Processed 1,250/60,000
Processed 1,500/60,000
Processed 1,750/60,000
Processed 2,000/60,000
Processed 2,250/60,000
Processed 2,500/60,000
Processed 2,750/60,000
Processed 3,000/60,000
Processed 3,250/60,000
Processed 3,500/60,000
Processed 3,750/60,000
Processed 4,000/60,000
Processed 4,250/60,000
Processed 4,500/60,000
Processed 4,750/60,000
Processed 5,000/60,000
Processed 5,250/60,000
Processed 5,500/60,000
Processed 5,750/60,000
Processed 6,000/60,000
Processed 6,250/60,000
Processed 6,500/60,000
Processed 6,750/60,000
Processed 7,000/60,000
Processed 7,250/60,000
Processed 7,500/60,000
Processed 7,750/60,000
Processed 8,000/60,000
Processed 8,250/60,000
Processed 8,500/60,000
Processed 8,750/60,000
Processed 9,000/60,000
Processed 9,250/60,000
Processed 9,500/60,000
Processed 9,750/60,000
Processed 10,000/60,000
Processed 10,250/60,000
Processed 10,500/60,000
Processed 10,750/60,000
Processed 11,

## Step 5 — Save new CSV files (original stays unchanged)

We create:
- `ArtBench-10-local-paths.csv` (full table with updated `url` column),
- `ArtBench-10-local-simple.csv` (WikiArt-like: `relative_path,label_id`),
- `ArtBench-10-download-failures.csv` (only if some downloads fail).

In [6]:
df_out = df.copy()
df_out['source_url'] = df_out['url']
df_out['url'] = result_df['local_path'].values
df_out['download_ok'] = result_df['ok'].values
df_out['download_error'] = result_df['error'].values

df_out.to_csv(OUTPUT_CSV, index=False)

failures = df_out[~df_out['download_ok']].copy()
if len(failures) > 0:
    failures.to_csv(FAILURES_CSV, index=False)
    print(f'Failure list saved to: {FAILURES_CSV}')
else:
    print('No failures found.')

label_order = sorted(df_out['label'].dropna().astype(str).unique())
label_to_id = {label: i for i, label in enumerate(label_order)}

simple_df = df_out[df_out['download_ok']].copy()
simple_df['label_id'] = simple_df['label'].astype(str).map(label_to_id)
simple_df[['url', 'label_id']].to_csv(SIMPLE_OUTPUT_CSV, index=False, header=False)

print(f'Full local CSV   : {OUTPUT_CSV}')
print(f'Simple local CSV : {SIMPLE_OUTPUT_CSV}')
print(f'Original unchanged: {CSV_PATH}')

display(simple_df[['url', 'label', 'label_id']].head(10))

Failure list saved to: C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10-download-failures.csv
Full local CSV   : C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10-local-paths.csv
Simple local CSV : C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10-local-simple.csv
Original unchanged: C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Artbench\ArtBench-10 (1).csv


,url,label,label_id
0,images/impressionism/frank-omeara_towards-nigh...,impressionism,3
1,images/impressionism/goldstein-grigoriy_mornin...,impressionism,3
2,images/impressionism/georges-lemmen_man-readin...,impressionism,3
3,images/impressionism/theodor-aman_port-of-cons...,impressionism,3
4,images/impressionism/niccolo-cannicci_il-passo...,impressionism,3
5,images/impressionism/howard-pyle_headpiece-for...,impressionism,3
6,images/impressionism/julian-alden-weir_nassau-...,impressionism,3
7,images/impressionism/ivan-vladimirov_rocks-of-...,impressionism,3
8,images/impressionism/laura-knight_a-dressing-r...,impressionism,3
9,images/impressionism/raoul-dufy_self-portrait-...,impressionism,3


## Done

After running all cells:
- images are in `datasets/Artbench/images/<label>/<filename>`,
- your original CSV is untouched,
- you have one full updated CSV + one WikiArt-like simple CSV.